# Fast panel solves

Thousands of panels by GMRES with fast multipole products: memory and time grow almost linearly with the panels, where the dense LU grows as their square and cube.

A round electrode at 1 V beside a square one at -1 V, with two free line charges, panelled ever finer. `solver="direct"` builds the dense matrix and factorises it. `solver="gmres"` never forms the matrix. Each product is an FMM sum over the panels' Gauss nodes plus the sparse exact near terms, and `"auto"` switches to it above `DIRECT_LIMIT` panels.

In [ ]:
import time

import numpy as np
import plasma_plots
from simgeometry import Circle, Polygon

from tree_d_3d import BoundaryIntegralElectrodes, conductors_from_shape
from tree_d_3d.electrodes import DIRECT_LIMIT

plasma_plots.set_backend("plotly")

box = Polygon(vertices=((1.5, -1.0), (3.0, -1.0), (3.0, 1.0), (1.5, 1.0)))
sources = np.array([[0.0, 1.2], [1.5, 0.0]])
charges = np.array([3e-11, -1e-11])  # C/m


def geometry(panel):
    circle = conductors_from_shape(Circle(midpoint=(0, 0), radius=1.0), 1.0, panel)
    return circle + conductors_from_shape(box, -1.0, panel)


print(f"auto iterates above {DIRECT_LIMIT} panels")
results = {"direct": [], "gmres": []}
for panel in (0.02, 0.01, 0.005, 0.0025, 0.00125):
    conductors = geometry(panel)
    for solver in ("direct", "gmres"):
        if solver == "direct" and panel < 0.005:
            continue  # the dense matrix grows as the square of the panels
        start = time.perf_counter()
        electrodes = BoundaryIntegralElectrodes(conductors, solver=solver)
        electrodes.solve(sources, charges)
        seconds = time.perf_counter() - start
        results[solver].append((electrodes.num_panels, seconds, electrodes))
        print(
            f"{solver:6s} {electrodes.num_panels:6d} panels: {seconds:6.2f} s, "
            f"{electrodes.iterations:3d} iterations, "
            f"charges {electrodes.total_charge()[0]:.6e} {electrodes.total_charge()[1]:.6e} C/m"
        )

Where both run, the two solves give the same panel charges. They differ by the GMRES tolerance times the condition number of the first-kind system.

In [ ]:
for (n, _, direct), (_, _, iterative) in zip(results["direct"], results["gmres"]):
    difference = (
        np.abs(iterative.charge - direct.charge).max() / np.abs(direct.charge).max()
    )
    print(f"{n:6d} panels: largest charge difference {difference:.1e}")

with plasma_plots.figure(1, 1) as fig:
    for solver, rows in results.items():
        n, seconds, _ = zip(*rows)
        fig[0].loglog(n, seconds, "o-", label=solver)
    fig[0].set_xlabel("panels")
    fig[0].set_ylabel("set-up and solve (s)")
    fig[0].legend()
fig